<a href="https://colab.research.google.com/github/ponkpook/nlp_practical_2025_sEXism/blob/feat%2Fnlp-task1.1/nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP Model Training for Sexism Detection (Sigmoid Output)

This notebook trains a Hugging Face transformer model to classify tweets as exhibiting hatred/sexism or not.
It uses the `text` column for input and `labels_task1_1` for the target, focusing on English language tweets.

The classification approach is as follows:
- The model is configured for binary classification with a single output logit.
- This logit is passed through a **sigmoid** function to obtain a probability (P(sexism)).
- If P(sexism) > 0.5, the tweet is classified as sexism.

## 1. Setup and Installations

In [ ]:
#!pip install datasets scikit-learn pandas torch torchvision torchaudio
!pip uninstall -y transformers accelerate peft
!pip install --upgrade pip
!pip install "transformers==4.39.3" "accelerate==0.30.0" "peft==0.10.0"

Found existing installation: transformers 4.39.3
Uninstalling transformers-4.39.3:
  Successfully uninstalled transformers-4.39.3
Found existing installation: accelerate 0.22.0
Uninstalling accelerate-0.22.0:
  Successfully uninstalled accelerate-0.22.0
Found existing installation: peft 0.10.0
Uninstalling peft-0.10.0:
  Successfully uninstalled peft-0.10.0
  Using cached transformers-4.39.3-py3-none-any.whl.metadata (134 kB)
  Using cached accelerate-0.30.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.10.0-py3-none-any.whl.metadata (13 kB)
Using cached transformers-4.39.3-py3-none-any.whl (8.8 MB)
Using cached accelerate-0.30.0-py3-none-any.whl (302 kB)
Using cached peft-0.10.0-py3-none-any.whl (199 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [peft]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 4.1.0 requires transfor

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import ast
import accelerate, transformers
from google.colab import drive
drive.mount('/content/drive')
# 文件路径就是 /content/drive/MyDrive/你的文件夹/文件名
print("accelerate:", accelerate.__version__)
print("transformers:", transformers.__version__)


Mounted at /content/drive
accelerate: 1.7.0
transformers: 4.52.4


## 3. Load and Prepare Data

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/EXIST2025_cleaned_plus_AEDA_fullinfo.csv')
print(f"Original dataset shape: {df.shape}")
df.head()

Original dataset shape: (13840, 18)


,id_EXIST,lang,tweet,number_annotators,annotators,gender_annotators,age_annotators,ethnicities_annotators,study_levels_annotators,countries_annotators,labels_task1_1,labels_task1_2,labels_task1_3,split,cleaned_text,aug_aeda,text,source
0,100001,es,"@TheChiflis Ignora al otro, es un capullo.El p...",6,"['Annotator_1', 'Annotator_2', 'Annotator_3', ...","['F', 'F', 'F', 'M', 'M', 'M']","['18-22', '23-45', '46+', '46+', '23-45', '18-...","['White or Caucasian', 'Hispano or Latino', 'W...","['Bachelor’s degree', 'Bachelor’s degree', 'Hi...","['Italy', 'Mexico', 'United States', 'Spain', ...","['YES', 'YES', 'NO', 'YES', 'YES', 'YES']","['REPORTED', 'JUDGEMENTAL', '-', 'REPORTED', '...","[['OBJECTIFICATION'], ['OBJECTIFICATION', 'SEX...",TRAIN_ES,"ignora al otro, es un capullo.el problema con ...","ignora al otro, es ? : un capullo.el problema ...","ignora al otro, es un capullo.el problema con ...",cleaned
1,100002,es,@ultimonomada_ Si comicsgate se parece en algo...,6,"['Annotator_7', 'Annotator_8', 'Annotator_9', ...","['F', 'F', 'F', 'M', 'M', 'M']","['18-22', '23-45', '46+', '46+', '23-45', '18-...","['Black or African American', 'Hispano or Lati...","['High school degree or equivalent', 'Bachelor...","['United Kingdom', 'Mexico', 'United States', ...","['NO', 'NO', 'NO', 'NO', 'YES', 'NO']","['-', '-', '-', '-', 'DIRECT', '-']","[['-'], ['-'], ['-'], ['-'], ['OBJECTIFICATION...",TRAIN_ES,si comicsgate se parece en algo a gamergate pu...,"si comicsgate . se parece , en algo a gamergat...",si comicsgate se parece en algo a gamergate pu...,cleaned
2,100003,es,"@Steven2897 Lee sobre Gamergate, y como eso ha...",6,"['Annotator_7', 'Annotator_8', 'Annotator_9', ...","['F', 'F', 'F', 'M', 'M', 'M']","['18-22', '23-45', '46+', '46+', '23-45', '18-...","['Black or African American', 'Hispano or Lati...","['High school degree or equivalent', 'Bachelor...","['United Kingdom', 'Mexico', 'United States', ...","['NO', 'NO', 'NO', 'NO', 'NO', 'NO']","['-', '-', '-', '-', '-', '-']","[['-'], ['-'], ['-'], ['-'], ['-'], ['-']]",TRAIN_ES,"lee sobre gamergate, y como eso ha cambiado la...","lee : sobre . gamergate, y como , ! eso ha cam...","lee sobre gamergate, y como eso ha cambiado la...",cleaned
3,100004,es,@Lunariita7 Un retraso social bastante lamenta...,6,"['Annotator_13', 'Annotator_14', 'Annotator_15...","['F', 'F', 'F', 'M', 'M', 'M']","['18-22', '23-45', '46+', '46+', '23-45', '18-...","['Hispano or Latino', 'Hispano or Latino', 'Wh...","['High school degree or equivalent', 'Bachelor...","['Mexico', 'Chile', 'Spain', 'Spain', 'Portuga...","['NO', 'NO', 'YES', 'NO', 'YES', 'YES']","['-', '-', 'DIRECT', '-', 'REPORTED', 'REPORTED']","[['-'], ['-'], ['IDEOLOGICAL-INEQUALITY'], ['-...",TRAIN_ES,"un retraso social bastante lamentable, gamerga...","un retraso social bastante lamentable, gamerga...","un retraso social bastante lamentable, gamerga...",cleaned
4,100005,es,@novadragon21 @icep4ck @TvDannyZ Entonces como...,6,"['Annotator_19', 'Annotator_20', 'Annotator_21...","['F', 'F', 'F', 'M', 'M', 'M']","['18-22', '23-45', '46+', '46+', '23-45', '18-...","['Hispano or Latino', 'Hispano or Latino', 'Wh...","['Bachelor’s degree', 'Bachelor’s degree', 'Ma...","['Mexico', 'Afghanistan', 'United States', 'It...","['YES', 'NO', 'YES', 'NO', 'YES', 'YES']","['REPORTED', '-', 'JUDGEMENTAL', '-', 'JUDGEME...","[['STEREOTYPING-DOMINANCE', 'OBJECTIFICATION']...",TRAIN_ES,entonces como así es el mercado lo mejor no es...,entonces como así es el mercado ! lo . ? mejor...,entonces como así es el mercado lo mejor no es...,cleaned


In [ ]:
import ast

# 1. 只保留英文推文和有效的 labels_task1_1
df_en = df[(df['lang'] == 'en') & (df['labels_task1_1'].notnull())].copy()
print(f"English tweets dataset shape: {df_en.shape}")

# 2. 保留文本和标签列
df_en = df_en[['text', 'labels_task1_1']].dropna()

# 3. 统计YES数量，超过3个YES为1，否则为0
def hard_label(label_str):
    try:
        labels = ast.literal_eval(label_str)
        yes_count = sum(x == 'YES' for x in labels)
        return 1 if yes_count > 3 else 0
    except Exception as e:
        print(f"标签解析失败: {label_str}, 错误: {e}")
        return None

df_en['label_hard'] = df_en['labels_task1_1'].apply(hard_label)
df_en = df_en[df_en['label_hard'].notnull()]
print(f"有效样本数：{len(df_en)}")
print(df_en[['text', 'labels_task1_1', 'label_hard']].head())

English tweets dataset shape: (6520, 18)
有效样本数：6520
                                                   text  \
3660  ffs! how about laying the blame on the bastard...   
3661  writing a uni essay in my local pub with a cof...   
3662  it is 2021 not 1921. i dont appreciate that on...   
3663  this is unacceptable. use her title as you did...   
3664  making yourself a harder target basically boil...   

                                 labels_task1_1  label_hard  
3660    ['YES', 'YES', 'NO', 'NO', 'YES', 'NO']           0  
3661  ['YES', 'YES', 'YES', 'NO', 'YES', 'YES']           1  
3662   ['YES', 'YES', 'NO', 'YES', 'NO', 'YES']           1  
3663    ['YES', 'YES', 'NO', 'YES', 'NO', 'NO']           0  
3664    ['YES', 'YES', 'NO', 'NO', 'NO', 'YES']           0  


## 4. Split Data

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_en['text'].tolist(),
    df_en['label_hard'].tolist(),   # 用 hard label
    test_size=0.2,
    random_state=42,
    stratify=df_en['label_hard']    # 用 hard label stratify
)

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")

Training samples: 5216
Validation samples: 1304


## 5. Tokenization and Dataset Creation

In [ ]:
MODEL_NAME = 'vinai/bertweet-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=512)

class SexismDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # 注意 labels 保持 float 类型
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)



train_dataset = SexismDataset(train_encodings, train_labels)
val_dataset = SexismDataset(val_encodings, val_labels)
print("训练集标签分布:")
print(pd.Series(train_labels).value_counts(normalize=True))

print("验证集标签分布:")
print(pd.Series(val_labels).value_counts(normalize=True))

config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/843k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.91M [00:00<?, ?B/s]

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


训练集标签分布:
0    0.651265
1    0.348735
Name: proportion, dtype: float64
验证集标签分布:
0    0.651074
1    0.348926
Name: proportion, dtype: float64


## 6. Model Training

In [ ]:
# For binary classification with sigmoid, num_labels should be 1.
# The model will output a single logit.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)

# def compute_metrics(pred):
#     labels = pred.label_ids
#     logits = pred.predictions # (batch, 1)
#     probs = torch.sigmoid(torch.from_numpy(logits)).numpy()
#     preds_classes = (probs.squeeze(-1) > 0.5).astype(int)
#     # 对于 soft label，计算硬标签下的各类指标
#     # 注意：labels 现在是 float（0~1），可用 0.5 进行离散化用于评估
#     hard_labels = (labels > 0.5).astype(int)
#     precision, recall, f1, _ = precision_recall_fscore_support(
#         hard_labels, preds_classes, average='binary', pos_label=1, zero_division=0
#     )
#     acc = accuracy_score(hard_labels, preds_classes)
#     return {
#         'accuracy': acc,
#         'f1': f1,
#         'precision': precision,
#         'recall': recall
#     }

def compute_metrics(pred):
    labels = pred.label_ids.astype(int)     # 已经是0/1，无需再二值化
    logits = pred.predictions
    probs = torch.sigmoid(torch.from_numpy(logits)).numpy()
    preds_classes = (probs.squeeze(-1) > 0.7).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds_classes, average='binary', pos_label=1, zero_division=0
    )
    acc = accuracy_score(labels, preds_classes)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

training_args = TrainingArguments(
    output_dir='./results_sigmoid',
    num_train_epochs=10,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_sigmoid',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.201200,0.180246,0.651074,0.000000,0.000000,0.000000
2,0.150200,0.122169,0.753834,0.484751,0.898810,0.331868
3,0.088000,0.101763,0.772239,0.536661,0.924731,0.378022
4,0.055900,0.069859,0.884969,0.813896,0.934473,0.720879
5,0.033500,0.056426,0.922546,0.883237,0.931707,0.839560
6,0.025200,0.050990,0.948620,0.925473,0.936937,0.914286
7,0.020900,0.059709,0.947086,0.925081,0.914163,0.936264
8,0.009700,0.039258,0.954755,0.933782,0.954128,0.914286
9,0.005900,0.039349,0.957055,0.938053,0.944321,0.931868
10,0.002800,0.038460,0.961656,0.945175,0.943107,0.947253


TrainOutput(global_step=820, training_loss=0.0647924512056861, metrics={'train_runtime': 491.8409, 'train_samples_per_second': 106.051, 'train_steps_per_second': 1.667, 'total_flos': 2439182026968960.0, 'train_loss': 0.0647924512056861, 'epoch': 10.0})

## 7. Evaluate Model

In [ ]:
eval_results = trainer.evaluate()
print("Evaluation Results (Sigmoid Model):")
for key, value in eval_results.items():
    print(f"  {key}: {value}")

Evaluation Results (Sigmoid Model):
  eval_loss: 0.03846029192209244
  eval_accuracy: 0.9616564417177914
  eval_f1: 0.9451754385964912
  eval_precision: 0.9431072210065645
  eval_recall: 0.9472527472527472
  eval_runtime: 1.7281
  eval_samples_per_second: 754.595
  eval_steps_per_second: 12.152
  epoch: 10.0


## 8. Prediction and Thresholding (Sigmoid Model)

To make predictions on new data:
1. Tokenize the input text.
2. Pass the tokenized input to the model.
3. The model outputs a single logit. **Apply a sigmoid function to this logit to convert it into a probability P(sexism).**
4. If P(sexism) > 0.5, classify as sexism.

In [ ]:
def predict_sexism_sigmoid(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    # Move inputs to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # Logits will be of shape (batch_size, 1) or just (1) for single input after squeeze
    logits = outputs.logits

    # Apply sigmoid to the single logit to get the probability of the positive class (sexism)
    # .item() converts a single-element tensor to a Python number
    sexism_probability = torch.sigmoid(logits).item()

    # Classify based on whether the probability of 'sexism' is > 0.5.
    is_sexism = sexism_probability > 0.5
    not_sexism_probability = 1.0 - sexism_probability

    return {
        "text": text,
        "is_sexism": bool(is_sexism),
        "sexism_probability": float(sexism_probability),
        "not_sexism_probability": float(not_sexism_probability)
    }

# Example usage (ensure the model is on the correct device)
if torch.cuda.is_available():
    model.to('cuda')
else:
    model.to('cpu')

sample_tweet_sexist = "Women belong in the kitchen, not in the office."
sample_tweet_not_sexist = "I had a great day today, the weather was lovely."

prediction1 = predict_sexism_sigmoid(sample_tweet_sexist, model, tokenizer)
print(f"Prediction for sexist tweet (Sigmoid): {prediction1}")

prediction2 = predict_sexism_sigmoid(sample_tweet_not_sexist, model, tokenizer)
print(f"Prediction for non-sexist tweet (Sigmoid): {prediction2}")

Prediction for sexist tweet (Sigmoid): {'text': 'Women belong in the kitchen, not in the office.', 'is_sexism': True, 'sexism_probability': 0.7411122918128967, 'not_sexism_probability': 0.25888770818710327}
Prediction for non-sexist tweet (Sigmoid): {'text': 'I had a great day today, girls needs to work hard on the weather was lovely.', 'is_sexism': True, 'sexism_probability': 0.5019168853759766, 'not_sexism_probability': 0.49808311462402344}


## 9. Save the model (Optional)

In [ ]:
model_save_path = "/content/drive/MyDrive/Colab Notebooks/saved_sexism_model_sigmoid"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Sigmoid model saved to {model_save_path}")


# from transformers import AutoTokenizer, AutoModelForSequenceClassification

# model_save_path = "/content/drive/MyDrive/Colab Notebooks/saved_sexism_model_sigmoid"
# model = AutoModelForSequenceClassification.from_pretrained(model_save_path)
# tokenizer = AutoTokenizer.from_pretrained(model_save_path)

Sigmoid model saved to /content/drive/MyDrive/Colab Notebooks/saved_sexism_model_sigmoid


10. Test the dev

In [ ]:
import ast

# 假设你的测试集csv里，label列叫 labels_task1_1，文本列叫 text
df_test = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/cleaned_dev_EXIST2025_training.csv')

# 1. 只保留英文推文和有效的 labels_task1_1
df_test_en = df_test[(df_test['lang'] == 'en') & (df_test['labels_task1_1'].notnull())].copy()
print(f"Test English tweets dataset shape: {df_test_en.shape}")

# 2. 保留文本和标签列
df_test_en = df_test_en[['cleaned_text', 'labels_task1_1']].dropna()

# 3. 统计YES数量，超过3个YES为1，否则为0
def hard_label(label_str):
    try:
        labels = ast.literal_eval(label_str)
        yes_count = sum(x == 'YES' for x in labels)
        return 1 if yes_count > 3 else 0
    except Exception as e:
        print(f"标签解析失败: {label_str}, 错误: {e}")
        return None

df_test_en['label_hard'] = df_test_en['labels_task1_1'].apply(hard_label)
df_test_en = df_test_en[df_test_en['label_hard'].notnull()]
print(f"测试集有效样本数：{len(df_test_en)}")
print(df_test_en[['cleaned_text', 'labels_task1_1', 'label_hard']].head())

# 4. 分词
test_texts = df_test_en['cleaned_text'].tolist()
test_labels = df_test_en['label_hard'].tolist()
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512)

# 5. 构建Dataset
class SexismDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)

test_dataset = SexismDataset(test_encodings, test_labels)
print("测试集标签分布:")
print(pd.Series(test_labels).value_counts(normalize=True))

Test English tweets dataset shape: (489, 15)
测试集有效样本数：489
                                          cleaned_text  \
549  you should smile more, love. just pretend your...   
550  she is right but the push is all in the opposi...   
551  everydaysexism some man moving my suitcase in ...   
552  lol gamergate the go to boogieman, maybe if th...   
553  to me this has the same negativity as gamergat...   

                                labels_task1_1  label_hard  
549     ['NO', 'NO', 'NO', 'NO', 'YES', 'YES']           0  
550   ['YES', 'YES', 'NO', 'YES', 'YES', 'NO']           1  
551  ['NO', 'YES', 'YES', 'YES', 'YES', 'YES']           1  
552      ['YES', 'NO', 'NO', 'NO', 'NO', 'NO']           0  
553     ['YES', 'NO', 'NO', 'YES', 'NO', 'NO']           0  
测试集标签分布:
0    0.603272
1    0.396728
Name: proportion, dtype: float64


In [ ]:
# 直接用 trainer.predict 预测
test_output = trainer.predict(test_dataset)
# test_output.metrics 里就是准确率、F1、精确率、召回率等
print("Test Results:")
for key, value in test_output.metrics.items():
    print(f"{key}: {value:.4f}")

# 或单独看预测概率和标签
probs = torch.sigmoid(torch.from_numpy(test_output.predictions)).numpy().squeeze(-1)
preds = (probs > 0.7).astype(int)  # 你的阈值
from sklearn.metrics import classification_report, confusion_matrix



Test Results:
test_loss: 0.1825
test_accuracy: 0.7853
test_f1: 0.7075
test_precision: 0.7697
test_recall: 0.6546
test_runtime: 0.5748
test_samples_per_second: 850.7870
test_steps_per_second: 13.9190


In [ ]:
# 取一条文本看看模型概率和预测
i = 0  # 第i条测试样本
print("文本:", texts[i])
print("真实标签:", labels[i])
print("模型判定:", preds[i])
print("模型判定为1概率:", probs[i])

NameError: name 'texts' is not defined